In [20]:
from typing import Dict, List, Any

def extract_labels(data: List[Dict[str, Any]]) -> Dict[str, Dict[str, str]]:
    """
    Извлекает метки и их значения из каждого словаря с уникальным id.
    
    Args:
        data: Список словарей, содержащих метки и их значения
        
    Returns:
        Словарь, где ключ - id, значение - словарь с метками и их значениями
        в формате {"label_name": "text"}
    """
    result = {}
    
    for item in data:
        if "id" not in item or "label" not in item:
            continue
            
        item_id = str(item["id"])
        labels = {}
        
        # Обрабатываем каждую метку в списке
        for label_item in item["label"]:
            if "text" in label_item and "labels" in label_item:
                text = label_item["text"]
                for label_name in label_item["labels"]:
                    if label_name not in labels:
                        labels[label_name] = [text]
                    else:
                        # Если метка уже существует, добавляем текст через запятую
                        labels[label_name].append(text)
                
        result[item_id] = labels
        
    return result 

In [24]:
import json

with open("../data/external/label-studio/label-example.json", "r", encoding="utf-8") as f:
    labels = json.load(f)

my_labels = extract_labels(labels)


In [22]:
with open("../data/processed/labels.json", "r", encoding="utf-8") as f:
    labels = json.load(f)

annotator_labels = extract_labels(labels)

In [29]:
annotator_labels

{'10': {'Brand': ['МИД России', 'РБК', 'Администрация', 'РБК']},
 '11': {'Brand': ['PwC', 'НАФИ', 'Ведомости'], 'Main': ['Ведомости']},
 '12': {'Brand': ['ТАСС', 'мэрии'], 'Main': ['ТАСС']},
 '13': {'Brand': ['Международного олимпийского комитета',
   'МОК',
   'Twitter',
   'Всемирного антидопингового агентства',
   'WADA',
   'МОК',
   'Федерации тяжелой атлетики России',
   'ФТАР',
   'WADA',
   'Российское антидопинговое агентство',
   'РУСАДА',
   'WADA'],
  'Main': ['Всемирного антидопингового агентства', 'WADA', 'WADA', 'WADA']},
 '14': {'Brand': ['Интерфакс', 'ТАСС'], 'Main': ['ТАСС']},
 '15': {'Brand': ['ТАСС',
   'Совете по развитию гражданского общества и правам человека',
   'СПЧ',
   'СПЧ',
   'СМИ',
   'СПЧ',
   'КПРФ'],
  'Main': ['ТАСС', 'КПРФ']},
 '16': {'Brand': ['Т— Ж',
   'vc.ru',
   'Тинькофф',
   'Facebook',
   'Нетологии',
   'Тинькофф',
   'Т— Ж',
   'Т— Ж'],
  'Main': ['Тинькофф']},
 '17': {'Brand': ['Facebook',
   'Газпрому',
   'Государственным бюро расследов

In [28]:
from sklearn.metrics import f1_score
import numpy as np

def calculate_f1_scores(true_labels, pred_labels):
    """
    Вычисляет macro и micro F1-score между двумя наборами меток.
    
    Args:
        true_labels: Словарь с истинными метками
        pred_labels: Словарь с предсказанными метками
        
    Returns:
        Словарь с macro и micro F1-score
    """
    # Находим общие id в обоих наборах
    common_ids = set(true_labels.keys()) & set(pred_labels.keys())
    
    if not common_ids:
        return {"macro_f1": 0.0, "micro_f1": 0.0, "common_ids_count": 0}
    
    # Собираем все уникальные метки
    all_labels = set()
    for id_val in common_ids:
        if id_val in true_labels:
            all_labels.update(true_labels[id_val].keys())
        if id_val in pred_labels:
            all_labels.update(pred_labels[id_val].keys())
    
    all_labels = sorted(list(all_labels))
    
    # Подготовка данных для вычисления F1-score
    y_true = []
    y_pred = []
    
    for id_val in common_ids:
        for label in all_labels:
            # Проверяем наличие метки в истинных данных
            true_has_label = label in true_labels[id_val]
            # Проверяем наличие метки в предсказанных данных
            pred_has_label = label in pred_labels[id_val]
            
            y_true.append(1 if true_has_label else 0)
            y_pred.append(1 if pred_has_label else 0)
    
    # Вычисляем F1-score
    macro_f1 = f1_score(y_true, y_pred, average='macro')
    micro_f1 = f1_score(y_true, y_pred, average='micro')
    
    return {
        "macro_f1": macro_f1,
        "micro_f1": micro_f1,
        "common_ids_count": len(common_ids)
    }

# Вычисляем F1-score между my_labels и annotator_labels
f1_scores = calculate_f1_scores(my_labels, annotator_labels)
print(f"Macro F1-score: {f1_scores['macro_f1']:.4f}")
print(f"Micro F1-score: {f1_scores['micro_f1']:.4f}")
print(f"Количество общих ID: {f1_scores['common_ids_count']}")



Macro F1-score: 0.5000
Micro F1-score: 0.6250
Количество общих ID: 8
